# Phase 2 on Kaggle: CIFAR-10 (Experiment B) + base temperature (Experiment C)
Settings: **GPU T4 x2**, **Internet On**. Then **Save Version -> Save & Run All (Commit)**.
Progress is printed every 5 minutes in the log. About 3.5 hours. Download `results_phase2.zip` from the Output tab.

In [ ]:
REPO_URL = "https://github.com/Abiha2004/lskd.git"
import os, subprocess, time
os.chdir("/kaggle/working")
!rm -rf lskd && git clone -q {REPO_URL} lskd
os.chdir("/kaggle/working/lskd")
!python scripts/download_teacher.py

In [ ]:
# GPU0: CIFAR-10 teacher -> CIFAR-10 KD -> tau=1      GPU1: tau=4 -> (waits for teacher) CIFAR-10 KD+LS
q0 = ("CUDA_VISIBLE_DEVICES=0 python train.py --preset c10_teacher_resnet56 --resume && "
      "CUDA_VISIBLE_DEVICES=0 python train.py --preset c10_kd --resume && "
      "CUDA_VISIBLE_DEVICES=0 python train.py --preset tau1_kd_ls --resume")
q1 = ("CUDA_VISIBLE_DEVICES=1 python train.py --preset tau4_kd_ls --resume && "
      "while [ ! -f runs/c10_teacher_resnet56/summary.json ]; do sleep 60; done && "
      "CUDA_VISIBLE_DEVICES=1 python train.py --preset c10_kd_ls --resume")
p0 = subprocess.Popen(q0 + " > gpu0.log 2>&1", shell=True)
p1 = subprocess.Popen(q1 + " > gpu1.log 2>&1", shell=True)
t0 = time.time()
def last(path):
    try:
        lines = [l for l in open(path).read().splitlines() if l.startswith("ep ") or l.startswith("Run ") or "Error" in l]
        return lines[-1][:110] if lines else "(starting)"
    except FileNotFoundError:
        return "(starting)"
while p0.poll() is None or p1.poll() is None:
    time.sleep(300)
    print(f"[{(time.time()-t0)/60:5.0f} min] GPU0: {last('gpu0.log')}", flush=True)
    print(f"[{(time.time()-t0)/60:5.0f} min] GPU1: {last('gpu1.log')}", flush=True)
print("exit codes", p0.returncode, p1.returncode)
!tail -n 5 gpu0.log gpu1.log

In [ ]:
!python evaluate.py --dataset cifar10 --teacher runs/c10_teacher_resnet56/final.pth --models "Vanilla KD=runs/c10_kd/final.pth" "KD+LS=runs/c10_kd_ls/final.pth"
!zip -q -r /kaggle/working/results_phase2.zip runs results gpu*.log -x "*/last.pth" "*_smoke/*"
!ls -lh /kaggle/working/results_phase2.zip